# NbNW Entropy Anomaly SETI — Colab Edition
## Free GPU Processing (T4/V100)

**Purpose:** Run entropy anomaly detection on cluster members via Google Colab.**Steps:** Install deps → Clone repo → Run analysis → Push results to GitHub

In [ ]:
#@title 🛠️ Install Dependencies
!pip install --quiet lightkurve astroquery numba numpy pandas scipy astropy
print("Dependencies installed ✅")

In [ ]:
#@title 🔗 Mount Drive + Clone Repo
from google.colab import drive
drive.mount("/content/drive")
!git clone https://github.com/ep150de/entropy-anomaly-seti.git
%cd /content/entropy-anomaly-seti
print("Environment ready ✅")

In [ ]:
#@title ⚙️ Load Pipeline
import numpy as np, pandas as pd, lightkurve as lk
from scipy.stats import ks_1samp
from concurrent.futures import ThreadPoolExecutor
import time, os

try:
    from numba import njit
    @njit(fastmath=True)
    def ks_ratio(flux):
        n=len(flux)
        if n<10: return 0.0
        fs=np.sort(flux)
        cdf_emp=np.arange(1,n+1)/n
        cdf_th=(fs-fs.min())/(fs.max()-fs.min()+1e-10)
        return float(np.max(np.abs(cdf_emp-cdf_th))*np.sqrt(n))
    NUMBA=True
    print("Numba JIT enabled ✅")
except:
    def ks_ratio(flux):
        return float(ks_1samp(flux,"norm").statistic*np.sqrt(len(flux)))
    NUMBA=False
    print("Scipy fallback ⚠️")

def process_star(tic_id, sector=11, window=150):
    try:
        search=lk.search_lightcurve(f"TIC {tic_id}", sector=sector)
        if len(search)==0: return {"tic_id":tic_id,"status":"no_data","anomalies":0,"mean_ks":0}
        lc=search.download_all().PDCSAP_FLUX.remove_nans().remove_outliers(sigma=5)
        flux=lc.flux.value
        n=len(flux)
        if n<window: window=min(100, n//2)
        nw=max(1, n//window)
        ks_vals=[ks_ratio(flux[i*window:(i+1)*window]) for i in range(nw)]
        if len(ks_vals)>10:
            mk=np.mean(ks_vals); std_k=np.std(ks_vals)
            thr=mk-3*std_k
            anom=int(sum(1 for k in ks_vals if k<thr))
        else:
            mk=np.mean(ks_vals) if ks_vals else 0; anom=0
        return {"tic_id":int(tic_id),"status":"ok","anomalies":anom,
                "mean_ks":float(mk),"std_ks":float(np.std(ks_vals) if ks_vals else 0)}
    except Exception as e:
        return {"tic_id":int(tic_id),"status":"error","error":str(e),"anomalies":0}

print("Pipeline loaded ✅")

In [ ]:
#@title 🚀 Run Parallel Analysis
catalog=pd.read_csv("data/expanded_star_catalog.csv")
targets=catalog["tic_id"].tolist()
print(f"Processing {len(targets)} targets...")

start=time.time(); results=[]
with ThreadPoolExecutor(max_workers=8) as ex:
    futs={ex.submit(process_star,t):t for t in targets}
    for i,f in enumerate(futs,1):
        r=f.result(); results.append(r)
        print(f"  [{i}/{len(targets)}] TIC {r["tic_id"]}: {r.get("anomalies",0)} anomalies")

rdf=pd.DataFrame(results)
rdf.to_csv("data/colab_results.csv",index=False)
print(f"
=== COMPLETE ===
Successful: {len(rdf[rdf["status"]=="ok"])}
")
print(f"Total anomalies: {rdf[rdf["status"]=="ok"]["anomalies"].sum()}
Time: {time.time()-start:.1f}s")
print("Saved to data/colab_results.csv")

In [ ]:
#@title 💾 Save to GitHub
!git config --global user.email "colab@nnw.ai"
!git config --global user.name "Colab Worker"
!git add data/colab_results.csv
!git commit -m "Colab analysis results"
!git push origin master
print("Results pushed ✅")

## Next Steps
1. Download results from Google Drive
2. Merge with workstation results
3. Proceed to cross-cluster correlation analysis